# MVP Demo: SWAP Test Fidelity with Hoeffding Planning

This notebook demonstrates the complete MVP:
- Hoeffding shot planning for bounded variables
- SWAP test circuit construction
- EstimatorV2 execution with noise modeling
- Coverage validation

In [1]:
import math
from qiskit_aer.noise import NoiseModel, depolarizing_error

from qamp_shotplanner import (
    HoeffdingPlanner,
    swap_test_1qubit,
    plan_shots_for_swap_fidelity,
    run_swap_fidelity_estimator,
    coverage_validation_swap,
)

## 1. Define States and Circuit

In [2]:
# Headline configuration from the report
theta1 = 0.3
theta2 = 0.8

qc = swap_test_1qubit(theta1, theta2)
print(f"Circuit: {qc.num_qubits} qubits, {qc.num_clbits} classical bits")
print(f"Data qubits: Ry({theta1}), Ry({theta2})")

Circuit: 3 qubits, 0 classical bits
Data qubits: Ry(0.3), Ry(0.8)


## 2. Compute Ideal Fidelity

In [3]:
# Analytic fidelity for Ry states
F_ideal = math.cos((theta1 - theta2) / 2) ** 2
print(f"Ideal (noiseless) fidelity: {F_ideal:.4f}")

Ideal (noiseless) fidelity: 0.9388


## 3. Plan Shots Using Hoeffding Bound

In [4]:
epsilon_F = 0.02   # tolerance on fidelity error
delta = 0.01       # failure probability

shots = plan_shots_for_swap_fidelity(epsilon_F, delta)
print(f"Planned shots (Hoeffding): {shots}")

Planned shots (Hoeffding): 26492


## 4. Build Noise Model

In [5]:
# Simple depolarizing noise model
noise_model = NoiseModel()
p_1q = 0.01  # 1% depolarizing on 1-qubit gates
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(p_1q, 1),
    ["ry", "rz", "sx", "x", "h"],
)
print(f"Noise model: {p_1q:.1%} depolarizing on 1-qubit gates")

Noise model: 1.0% depolarizing on 1-qubit gates


## 5. Three-Level Error Decomposition

In [6]:
# High-precision noisy reference (what the device "really" does)
reference_shots = 100000  # smaller than 1M for speed, can be increased
F_noisy_ref = run_swap_fidelity_estimator(
    qc,
    shots=reference_shots,
    noise_model=noise_model,
    seed_simulator=9999,
)

# Hoeffding-planned run
F_hat = run_swap_fidelity_estimator(
    qc,
    shots=shots,
    noise_model=noise_model,
    seed_simulator=1234,
)

statistical_error = abs(F_hat - F_noisy_ref)
hardware_bias = abs(F_noisy_ref - F_ideal)
total_error = abs(F_hat - F_ideal)

print("\n=== SWAP-Test Fidelity Under Noisy Channel ===")
print(f"Ideal fidelity (F_ideal):      {F_ideal:.4f}")
print(f"Noisy reference (F_device):    {F_noisy_ref:.4f}  ({reference_shots:,} shots)")
print(f"Hoeffding estimate (F_hat):    {F_hat:.4f}  ({shots:,} shots)")
print(f"\nError Decomposition:")
print(f"  Statistical: |F_hat - F_device|    = {statistical_error:.4f}")
print(f"  Hardware bias: |F_device - F_ideal| = {hardware_bias:.4f}")
print(f"  Total error: |F_hat - F_ideal|     = {total_error:.4f}")
print(f"\nTarget statistical error: {epsilon_F:.4f}")
print(f"Actual statistical error: {statistical_error:.4f}")
if statistical_error <= epsilon_F:
    print("✓ Statistical bound satisfied")
else:
    print("✗ Statistical bound violated")


=== SWAP-Test Fidelity Under Noisy Channel ===
Ideal fidelity (F_ideal):      0.9388
Noisy reference (F_device):    0.8180  (100,000 shots)
Hoeffding estimate (F_hat):    0.8186  (26,492 shots)

Error Decomposition:
  Statistical: |F_hat - F_device|    = 0.0006
  Hardware bias: |F_device - F_ideal| = 0.1208
  Total error: |F_hat - F_ideal|     = 0.1202

Target statistical error: 0.0200
Actual statistical error: 0.0006
✓ Statistical bound satisfied


## 6. Coverage Validation

Run multiple trials to empirically verify the Hoeffding bound.

In [7]:
# Coverage validation parameters
n_trials = 100  # small for speed; increase to 1000 for "paper-grade" validation

print(f"Running {n_trials} trials with {shots:,} shots each...")
print(f"Expected failure rate: {delta:.1%}")
print(f"Tolerance: {epsilon_F:.4f}\n")

stats = coverage_validation_swap(
    theta1=theta1,
    theta2=theta2,
    n_trials=n_trials,
    epsilon_F=epsilon_F,
    delta=delta,
    reference_shots=reference_shots,
    noise_model=noise_model,
)

print("\n" + "=" * 50)
print("COVERAGE VALIDATION RESULTS")
print("=" * 50)
print(f"Trials: {stats.n_trials}")
print(f"Shots/trial: {stats.shots_per_trial:,}")
print(f"\nBound violations: {stats.failures} / {stats.n_trials}")
print(f"Empirical failure rate: {stats.empirical_failure_rate:.2%}")
print(f"Theoretical bound (delta): {stats.delta:.2%}")
print(f"\nError statistics:")
print(f"  Mean error: {stats.mean_error:.6f}")
print(f"  Max error: {stats.max_error:.6f}")
print(f"  Tolerance (epsilon_F): {stats.epsilon_F:.6f}")

if stats.empirical_failure_rate <= delta * 1.5:
    print(f"\n✓ Validation PASSED")
else:
    print(f"\n✗ Validation WARNING")

Running 100 trials with 26,492 shots each...
Expected failure rate: 1.0%
Tolerance: 0.0200


COVERAGE VALIDATION RESULTS
Trials: 100
Shots/trial: 26,492

Bound violations: 0 / 100
Empirical failure rate: 0.00%
Theoretical bound (delta): 1.00%

Error statistics:
  Mean error: 0.000580
  Max error: 0.000614
  Tolerance (epsilon_F): 0.020000

✓ Validation PASSED
